In [ ]:
import torch
import torch.nn as nn
import torchvision
import albumentations as A
from torch.utils.data import Dataset, DataLoader
from torch.autograd import Variable
from torchvision.models import segmentation as seg_models
from torchvision import transforms as T
from torchvision.transforms import functional as F
# from unet import UNet
from tqdm import tqdm
from torchsummary import summary

import time
import os
import numpy as np
import cv2
from PIL import Image
from typing import List, Dict, Tuple, Union, Final
import sys
sys.path.append('../utils')
from json_parser import CellMaskDataset

## Configurations
### Model parameters

In [ ]:
# number of classes (including background)
NUM_CLASSES: Final[int] = 2
# model input image large/small-side sizes
MODEL_INPUT_MAX_SIZE: Final[int] = 1024
MODEL_INPUT_MIN_SIZE: Final[int] = 1024 

### Training parameters

In [ ]:
# training batch size
# this should be at least 2 as the DeepLab model Batch norm require at least 2
BATCH_SIZE: Final[int] = 4
# learning rate
LEARNING_RATE: Final[float] = 1e-3
# number of training epochs
NUM_EPOCHS = 10
# learning rate decay steps, a value of 0 means One-cycle LR scheduler should be used
LR_DECAY_STEPS = 0

### Train/Test image and annotation folders

In [ ]:
# the location of the images and json annotations 
# json annotations are instance annotations in Darwin7 2.0 format (not semantic segments)
# the data model extract the semantic masks from these annotations
# the image and annotation folders should include all (train+test) images and annotations
IMAGES_PATH = '/home/cellareye/Cellanome/Data/cytoplasm_bf_images'
ANNOTS_PATH = '/home/cellareye/Cellanome/Data/cytoplasm_bf_annotations'

# the location of two test.txt and train.txt files listing the names (excluding extensions)
# of the images assigned to each set
TRAIN_TEST_SPLIT_FOLDERS = ['/home/cellareye/Cellanome/Data/new_datasets/230607_IMR90_training_dataset_1_cytoplasm_bf', 
                            '/home/cellareye/Cellanome/Data/new_datasets/230622_IMR90_training_dataset_3_cytoplasm_bf',
                            '/home/cellareye/Cellanome/Data/new_datasets/230915_IMR90_training_dataset_3_cytoplasm', 
                           #  '/home/cellareye/Cellanome/Data/normalised-12212022-tregs-beads-cages-bb2-not-reviewed'
                           ]
                            

test_annotation_files: List[str] = []
train_annotation_files: List[str] = []
for folder in TRAIN_TEST_SPLIT_FOLDERS:
    with open(os.path.join(folder, 'test.txt')) as file:
        filenames = file.readlines()
        filenames = [f.replace('\n', '') for f in filenames if len(f) > 0]
        test_annotation_files += filenames

    with open(os.path.join(folder, 'train.txt')) as file:
        filenames = file.readlines()
        filenames = [f.replace('\n', '') for f in filenames if len(f) > 0]
        train_annotation_files += filenames

LABEL_MAP: Dict[int, str] = {1: 'cytoplasm'}
REVERSE_LABEL_MAP: Dict[str, int] = {'cytoplasm': 1}
    
MODEL_PATH = 'checkpoints'
if not os.path.exists(MODEL_PATH):
    os.mkdir(MODEL_PATH)

## Data Model 
### Dataset class

In [ ]:
class CytoplasmDataset(Dataset):
    
    def __init__(self, 
                 images_path: str, 
                 annotations_path: str,
                 annotations: List[str],
                 reverse_label_map: Dict[str, int],
                 mean: List[float] = [0.449], 
                 std: List[float] = [0.226], 
                 color_depth: int = 8,
                 min_object_diameter: float = 6.0,
                 max_larger_side: int = 2048,
                 max_smaller_side: int = 2048,
                 transform = None, 
                 patch_size: int = -1):
            
        
    
        self.parsed_json = CellMaskDataset(images_path=images_path, 
                                           annotations_path=annotations_path, 
                                           annotations=annotations,
                                           labels_of_interest=list(reverse_label_map.keys()), 
                                           percentage_to_expand_bbox_boundaries=0.0, 
                                           color_depth=color_depth, 
                                           min_object_diameter=min_object_diameter,
                                           max_larger_side=max_larger_side, 
                                           max_smaller_side=max_larger_side,
                                           normalize=False,
                                           class_names_to_ids_map=reverse_label_map)
    
        
        self.transform = transform
        self.patch_size: int = patch_size
        self.mean: List[float] = mean
        self.std: List[float] = std
        
    
    def __len__(self) -> int:
        return len(self.parsed_json)
    
    def __getitem__(self, idx: int) -> (torch.Tensor, torch.Tensor):
        
        data: dict = self.parsed_json[idx]
        # if image has 3 channels, it will be in RGB
        image: np.ndarray = data['image']
        image_height, image_width = image.shape[:2]
        
        # the model expects a 3 channel image, 
        image = np.repeat(np.expand_dims(image, axis=2), 3, axis=2)
        
        # semantic mask in full image resolution
        # we are using np.uint8, hence only 255 segments 
        semantic_mask: np.ndarray = np.zeros((image_height, image_width), np.uint8)
        for row_idx, row in data['annotations'].iterrows():
            xmin, ymin, xmax, ymax = row[['xtl', 'ytl', 'xbr', 'ybr']]
            # no need to check the validity 
            if xmin >= xmax or ymin >= ymax:
                continue
                
            label = int(row['label'])    
            # update the semantic mask
            instance_mask: np.ndarray = data['masks'][row_idx] * label 
            
            # only update the non-zero areas, otherwise, we may remove parts of
            # the semantic mask from other instances
            semantic_mask[ymin:ymax, xmin:xmax][instance_mask > 0] = instance_mask[instance_mask > 0]
             
        
        if self.transform is not None:
            augmented = self.transform(image=image, mask=semantic_mask)
            image = augmented['image']
            semantic_mask = augmented['mask']
        
        # convert the image (numpy array) to a torch Tensor and normalize it
        convert_normalize_t = T.Compose([T.ToTensor(), T.Normalize(self.mean, self.std)])
        image_tensor: torch.Tensor = convert_normalize_t(image)
        mask_tensor: torch.Tensor = torch.from_numpy(semantic_mask).long()
        
        if self.patch_size > 0:
            image_tensor, mask_tensor = self.tiles(image_tensor, mask_tensor)
        
        return image_tensor, mask_tensor
    
    def tiles(self, img, mask) -> (torch.Tensor, torch.Tensor):

        img_patches = img.unfold(1, self.patch_size, self.patch_size).unfold(2, self.patch_size, self.patch_size) 
        img_patches  = img_patches.contiguous().view(img.shape[0], -1, self.patch_size, self.patch_size) 
        img_patches = img_patches.permute(1, 0, 2, 3)
        
        mask_patches = mask.unfold(0, self.patch_size, self.patch_size).unfold(1, self.patch_size, self.patch_size)
        mask_patches = mask_patches.contiguous().view(-1, self.patch_size, self.patch_size)
        
        return img_patches, mask_patches

### Image and segmentation mask transforms
Here, we use albumentations package that takes both image and annotations to apply the transformations on them. It takes and returns numpy arrays. 

In [ ]:
train_transform = A.Compose([A.HorizontalFlip(), 
                             A.VerticalFlip(), 
                             A.GridDistortion(p=0.2), 
                             A.RandomBrightnessContrast((0, 0.5), (0, 0.5)),
                             A.GaussNoise()])

### Datasets and dataloaders

In [ ]:
# datasets
train_dataset = CytoplasmDataset(images_path=IMAGES_PATH, 
                                 annotations_path=ANNOTS_PATH,
                                 annotations=train_annotation_files,
                                 reverse_label_map={'cytoplasm': 1},
                                 transform = train_transform, 
                                 max_larger_side=MODEL_INPUT_MAX_SIZE,
                                 max_smaller_side=MODEL_INPUT_MIN_SIZE)

test_dataset = CytoplasmDataset(images_path=IMAGES_PATH, 
                                annotations_path=ANNOTS_PATH,
                                annotations=test_annotation_files,
                                reverse_label_map={'cytoplasm': 1}, 
                                max_larger_side=MODEL_INPUT_MAX_SIZE,
                                max_smaller_side=MODEL_INPUT_MIN_SIZE)

# dataloaders
# drop_last is set to True to avoid passing a data with batch size of 1
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)     

### Visualization

In [ ]:
def to_numpy(tensor: torch.Tensor) -> np.ndarray:
    """
    A function to convert a torch input to numpy array.
    Args:
        tensor (torch tensor).
    Returns:
        Converted to numpy array.
    """
    return tensor.detach().cpu().numpy() if tensor.requires_grad else tensor.cpu().numpy()


# module-level variables and constants
OPEN_CV_BLACK: Final[Tuple[int, int, int]] = (0, 0, 0)
OPEN_CV_WHITE: Final[Tuple[int, int, int]] = (255, 255, 255)
OPEN_CV_RED: Final[Tuple[int, int, int]] = (0, 0, 255)
OPEN_CV_BLUE: Final[Tuple[int, int, int]] = (255, 0, 0)
OPEN_CV_GREEN: Final[Tuple[int, int, int]] = (0, 195, 0)
OPEN_CV_BRIGHT_GREEN: Final[Tuple[int, int, int]] = (0, 255, 0)
OPEN_CV_MAGENTA: Final[Tuple[int, int, int]] = (255, 0, 255)
OPEN_CV_YELLOW: Final[Tuple[int, int, int]] = (0, 255, 255)
OPEN_CV_CYAN: Final[Tuple[int, int, int]] = (255, 255, 0)
OPEN_CV_ORANGE: Final[Tuple[int, int, int]] = (0, 165, 255)
OPEN_CV_GRAY: Final[Tuple[int, int, int]] = (169, 169, 169)
GREEN_COLOR_MULTIPLIER: Final[Tuple[float, float, float]] = (0.6, 1.0, 0.6)
# all colors in a specific order for debug image coloring
COLORS: Final[List[Tuple[int, int, int]]] = [
    OPEN_CV_BLACK,
    OPEN_CV_GREEN,
    OPEN_CV_WHITE,
    OPEN_CV_BLUE,
    OPEN_CV_RED,
    OPEN_CV_BRIGHT_GREEN,
    OPEN_CV_MAGENTA,
    OPEN_CV_CYAN,
    OPEN_CV_YELLOW]

def show_sample(idx: int, train: bool=True):
    if train:
        img_t, mask_t = train_dataset[idx]
    else:
        img_t, mask_t = test_dataset[idx]
    
    image: np.ndarray = to_numpy(img_t.permute(1, 2, 0).squeeze())
    mask: np.ndarray = to_numpy(mask_t).astype(np.uint8)
    
    # scale back and add the mean, scale to 0-255
    image = ((image * train_dataset.std + train_dataset.mean) * 255).astype(np.uint8)
    
    class_ids: List[int] = np.unique(mask)[1:]
    if len(image.shape) < 3:
        image = np.repeat(np.expand_dims(image, axis=2), 3, axis=2)
    
    mask = np.repeat(np.expand_dims(mask, axis=2), 3, axis=2)
    
    for class_id in class_ids:
        class_pixels: Tuple[np.ndarray, np.ndarray] = np.where(mask==class_id)
        image[class_pixels] = 0.6 * image[class_pixels] + 0.4 * (mask * COLORS[class_id])[class_pixels]
    
    return image.astype(np.uint8)

In [ ]:
img = show_sample(3, True)
Image.fromarray(img[:, :, ::-1])

## Model definition

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
# model = seg_models.deeplabv3_resnet50(weights=seg_models.DeepLabV3_ResNet50_Weights.DEFAULT) # Load net
model = seg_models.fcn_resnet50(weights=seg_models.FCN_ResNet50_Weights.DEFAULT)

# Change final layer to 2 classes 
# note that FCN and DeepLab models have different number of channels for this layer (512 vs 256 for DeepLab)
num_channels: int =  model.classifier[4].state_dict()['weight'].shape[1]
model.classifier[4] = torch.nn.Conv2d(num_channels, NUM_CLASSES, kernel_size=(1, 1), stride=(1, 1)) 

# UNet model
# model = smp.Unet('mobilenet_v2', encoder_weights='imagenet', 
#                  classes=23, activation=None, 
#                  encoder_depth=5, 
#                  decoder_channels=[256, 128, 64, 32, 16])

In [ ]:
# device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
# model = UNet(num_classes=NUM_CLASSES, out_size=(MODEL_INPUT_MIN_SIZE,  MODEL_INPUT_MAX_SIZE))

## Training
### Optimizer, LR scheduler and loss function

In [ ]:
# construct an optimizer
# Adam optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(params, lr = LEARNING_RATE)

print(f"Adam Optimizer is configured for {NUM_EPOCHS} epochs")

print(f"Initial learning rate is set to {LEARNING_RATE}")
if LR_DECAY_STEPS < 1:
    print(f"One-Cyle LR scheduler is configured for {NUM_EPOCHS} with {len(train_loader)} steps per epoch")
    lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer=optimizer, 
                                                       max_lr=LEARNING_RATE, 
                                                       epochs=NUM_EPOCHS,
                                                       steps_per_epoch=len(train_loader))
    
else:
    print(f"Step LR scheduler is configured with {LR_DECAY_STEPS} epochs for each step")
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer=optimizer,
                                                   step_size=LR_DECAY_STEPS,
                                                   gamma=0.1)



criterion = nn.CrossEntropyLoss()

### Performance metrics

In [ ]:
def pixel_accuracy(output: torch.Tensor, mask: torch.Tensor) -> float:
    """
    Pixel-wise accuracy
    
    Args:
        output: BATCH_SIZE * NUM_CLASSES * H * W tensor of class predictions for each pixel
        mask: H * W tensor of semantic mask
    Returns:
        Accuracy
    """
    with torch.no_grad():
        output: torch.Tensor = torch.argmax(torch.nn.functional.softmax(output, dim=1), dim=1)
        correct: torch.Tensor = torch.eq(output, mask).int()
        accuracy = float(correct.sum()) / float(correct.numel())
    return accuracy

def mIoU(pred_mask: torch.Tensor, 
         mask: torch.Tensor, 
         smooth: float = 1e-10, 
         num_classes: int = NUM_CLASSES):
    """
    Mean IoU (among different classes)
    
    Args:
        output: H * W tensor of predicted semantic segmentation masl
        mask: H * W tensor of semantic mask
        smooth: A small float to avoid divide by zero
        num_classes: Number of classes
    Returns:
        Average IoU over all classes
    """
    with torch.no_grad():
        pred_mask = torch.nn.functional.softmax(pred_mask, dim=1)
        pred_mask = torch.argmax(pred_mask, dim=1)
        pred_mask = pred_mask.contiguous().view(-1)
        mask = mask.contiguous().view(-1)

        iou_per_class = []
        for class_id in range(num_classes): #loop per pixel class
            true_class: torch.Tensor = pred_mask == class_id
            true_label: torch.Tensor = mask == class_id

            if true_label.long().sum().item() == 0:
                # the class does not exist in this mask
                iou_per_class.append(np.nan)
            else:
                intersect = torch.logical_and(true_class, true_label).sum().float().item()
                union = torch.logical_or(true_class, true_label).sum().float().item()

                iou = (intersect + smooth) / (union + smooth)
                iou_per_class.append(iou)
        return np.nanmean(iou_per_class)

### Training script

In [ ]:
def train(model, 
          train_loader, 
          test_loader, 
          criterion, 
          optimizer, 
          lr_scheduler,
          num_epochs,
          device,
          patch=False):
    
    torch.cuda.empty_cache()
    
    # losses, accuracies and mean IoUs over the training epochs
    train_losses: List[float] = []
    test_losses: List[float] = []
    train_ious: List[float] = []
    train_accs: List[float] = []
    test_ious: List[float] = [] 
    test_accs: List[float] = []
    
    # learning rates used for each step (not each epoch as we may use OneCyle scheduling)
    lrs: List[float] = []
    min_loss: float = np.inf
    
    decrease = 1 
    not_improve = 0
    
    model = model.to(device)
    
    start_time = time.time()
    
    for epoch in range(num_epochs):
        
        since = time.time()
        
        running_loss: float = 0
        iou_score: float = 0
        accuracy: float = 0
        
        # training loop
        model.train()
        for i, data in enumerate(tqdm(train_loader)):
            # training phase
            image_tiles, mask_tiles = data
            if patch:
                bs, n_tiles, c, h, w = image_tiles.size()

                image_tiles = image_tiles.view(-1, c, h, w)
                mask_tiles = mask_tiles.view(-1, h, w)
            
            image = image_tiles.to(device) 
            mask = mask_tiles.to(device)
            # forward
            output = model(image)['out']
            loss = criterion(output, mask)
            # evaluate metrics
            iou_score += mIoU(output, mask)
            accuracy += pixel_accuracy(output, mask)
            # backward
            loss.backward()
            optimizer.step() # update weight          
            optimizer.zero_grad() # reset gradient
            
            # update the learning rate only after one batch in case of One-Cycle LR scheduler
            lrs.append(lr_scheduler.get_last_lr()[0])
            if isinstance(lr_scheduler, torch.optim.lr_scheduler.OneCycleLR):
                lr_scheduler.step() 
            
            running_loss += loss.item()
        
        # update the learning rate after one full epoch if LR step scheduler is used
        if isinstance(lr_scheduler, torch.optim.lr_scheduler.StepLR):
            lr_scheduler.step() 
        
        # run the validation after each training epoch
        model.eval()
        test_running_loss: float = 0
        test_iou_score: float = 0
        test_accuracy: float = 0
        
        # validation loop
        with torch.no_grad():
            for i, data in enumerate(tqdm(test_loader)):
                # reshape to 9 patches from single image, delete batch size
                image_tiles, mask_tiles = data

                if patch:
                    bs, n_tiles, c, h, w = image_tiles.size()
                    image_tiles = image_tiles.view(-1, c, h, w)
                    mask_tiles = mask_tiles.view(-1, h, w)
                    
                image = image_tiles.to(device) 
                mask = mask_tiles.to(device)
                output = model(image)['out']
                # evaluation metrics
                test_iou_score +=  mIoU(output, mask)
                test_accuracy += pixel_accuracy(output, mask)
                # loss
                loss = criterion(output, mask)                                  
                test_running_loss += loss.item()
            
        # calculatio mean for each batch
        running_loss /= len(train_loader)
        iou_score /= len(train_loader)
        accuracy /= len(train_loader)
        
        test_running_loss /= len(test_loader)
        test_iou_score /= len(test_loader)
        test_accuracy /= len(test_loader)
                       
         # save the results
        train_losses.append(running_loss)
        train_ious.append(iou_score)
        train_accs.append(accuracy)
        
        test_losses.append(test_running_loss)
        test_ious.append(test_iou_score)
        test_accs.append(test_accuracy)
        
        print('saving the model ...')
        torch.save(model.state_dict(), os.path.join(MODEL_PATH, 'checkpoint_' + str(epoch) +'.pt'))
                    
        
        print("Epoch:{}/{} ... \n".format(epoch + 1, num_epochs),
              "Train Loss: {:.3f} \n".format(running_loss),
              "Test Loss: {:.3f} \n".format(test_running_loss),
              "Train mean IoU: {:.3f} \n".format(iou_score),
              "Test mean IoU: {:.3f} \n".format(test_iou_score),
              "Train Accuracy: {:.3f} \n".format(accuracy),
              "Test Accuracy: {:.3f} \n".format(test_accuracy),
              "Time: {:.2f} m".format((time.time() - since) / 60))
        
    history = {'train_loss' : train_losses, 'test_loss': test_losses,
               'train_mean_iou' :train_ious, 'test_mean_iou': test_ious,
               'train_acc': train_accs, 'val_acc': test_accs,
               'lrs': lrs}
    print('Total time: {:.2f} m' .format((time.time()- start_time) / 60))
    return history

In [ ]:
history  = train(model, 
                 train_loader, 
                 test_loader, 
                 criterion, 
                 optimizer, 
                 lr_scheduler,
                 NUM_EPOCHS,
                 device,
                 patch=False)

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
# model = seg_models.deeplabv3_resnet50(weights=seg_models.DeepLabV3_ResNet50_Weights.DEFAULT)
model = seg_models.fcn_resnet50(weights=seg_models.FCN_ResNet50_Weights.DEFAULT)
# model = seg_models.deeplabv3_mobilenet_v3_large(weights=seg_models.DeepLabV3_MobileNet_V3_Large_Weights.DEFAULT)

# Change final layer to 2 classes 
# note that FCN and DeepLab models have different number of channels for this layer (512 vs 256 for DeepLab)
num_channels: int =  model.classifier[4].state_dict()['weight'].shape[1]
model.classifier[4] = torch.nn.Conv2d(num_channels, NUM_CLASSES, kernel_size=(1, 1), stride=(1, 1)) 

model.load_state_dict(torch.load(os.path.join(MODEL_PATH, 'bf_fcn_resnet50_1cycle_lrs_4_bs_10_epochs_2.pt')))

In [ ]:
def evaluate(model, data_loader, device, patch=False):
    # run the validation after each training epoch
    model.eval()
    model.to(device)
    test_iou_score: float = 0
    test_accuracy: float = 0
    
    # validation loop
    with torch.no_grad():
        for i, data in enumerate(tqdm(data_loader)):
            # reshape to 9 patches from single image, delete batch size
            image_tiles, mask_tiles = data
            if patch:
                bs, n_tiles, c, h, w = image_tiles.size()
                image_tiles = image_tiles.view(-1, c, h, w)
                mask_tiles = mask_tiles.view(-1, h, w)
                
            image = image_tiles.to(device) 
            mask = mask_tiles.to(device)
            output = model(image)['out']
            # evaluation metrics
            test_iou_score +=  mIoU(output, mask)
            test_accuracy += pixel_accuracy(output, mask)
            
    
    test_iou_score /= len(data_loader)
    test_accuracy /= len(data_loader)
                   
    
    print("Test mean IoU: {:.3f} \n".format(test_iou_score),
          "Test Accuracy: {:.3f} \n".format(test_accuracy))

In [ ]:
evaluate(model, test_loader, device)

In [ ]:
def predict(model, image, device):
    
    mean: Final[float] = 0.449 
    std: Final[float] = 0.226 
    # make sure the image is a gray scale image (BGR or RGB does not matter)
    # then normalize
    if len(image.shape) == 3:
        img = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        img = (image / 255.0 - mean) / std
    else:
        img = (image / 255.0 - mean) / std
        
    # convert to tensor and add batch dimension
    img = np.repeat(np.expand_dims(img, axis=2), 3, axis=2)
    
    image_tensor = F.to_tensor(img).unsqueeze(dim=0).to(device).float()
    
    model.eval()
    model.to(device)
    
    with torch.no_grad():
        output = model(image_tensor)["out"]
        mask = torch.argmax(output, dim=1).squeeze().cpu().numpy()
    return mask.astype(np.uint8)

In [ ]:
mask = predict(model, test_dataset.parsed_json[3]['image'], device)

In [ ]:
Image.fromarray(show_sample(3, train=False))

In [ ]:
Image.fromarray(mask * 255)

In [ ]:
import time
start = time.time()
for i in range(100):
    mask = predict(model, test_dataset.parsed_json[1]['image'], device)
print(f"Model runtime took {np.round((time.time() - start) * 10, 2)} ms")

In [ ]:
import os
os.listdir('checkpoints')

In [ ]:
# old model, old test set
# Test mean IoU: 0.829 
# Test Accuracy: 0.967

# new model, old test set
# Test mean IoU: 0.827 
# Test Accuracy: 0.968

# old model, new test set (more cytoplasm annotations and images with cages but no cytoplasm) 
# Test mean IoU: 0.878 
# Test Accuracy: 0.954    

# new model, new test set
# Test mean IoU: 0.930 
# Test Accuracy: 0.989